# STT(Speech To Text)
- 사람의 말을 문자로 바꿔주는 기술
- 예시
    - 유튜브 자막 자동 생성
    - 스마트폰 음성 입력: “카카오톡 보내줘~” 말하면 자동 입력됨
    - 콜센터 녹취 자동 기록
    - 회의 녹음 → 회의록 자동 생성

## 1. SpeechRecognition(음성인식) 

- 여러 음성 인식 엔진을 파이썬에서 쉽게 호출하게 해주는 래퍼 라이브러리
- 예제에서는 Google Web Speech API와 OpenAI 인식 함수를 사용해 WAV 파일 또는 마이크 입력을 텍스트로 변환함

### 1) SpeechRecognition(음성인식) 패키지 설치

In [21]:
%pip install SpeechRecognition pydub pyaudio

Note: you may need to restart the kernel to use updated packages.


### 2) mp3 파일을 .wav로 변환하기
- gTTS는 mp3 파일만 생성하지만 SpeechRecognition 라이브러리는 .wav 확장자 사운드 파일만 인식할 수 있음

In [ ]:
from pydub import AudioSegment

mp3_file = "./audio/sample.mp3"
wav_file = "./audio/sample.wav"

sound = AudioSegment.from_mp3(mp3_file)
sound.export(wav_file, format="wav", parameters=["-ac", "1", "-ar", "16000"]) # 오디오 채널 mono 변환, 샘플링 레이트 16kHz 변환

### 3) wav 파일로부터 STT

In [28]:
from pydub import AudioSegment
import speech_recognition as sr

# 음성 인식 객체 생성
recognizer = sr.Recognizer()

audio_file = "./audio/sample.wav"

with sr.AudioFile(audio_file) as sourcs:
    print("음성을 인식 중입니다....")
    audio_data = recognizer.record(sourcs)

text = recognizer.recognize_google(audio_data, language='en')
print("변환된 텍스트 : ", text)

음성을 인식 중입니다....
변환된 텍스트 :  Hello nice to meet you today is a great day


### 4) 실시간 녹음과 STT

In [29]:
with sr.Microphone() as source:
    print("녹음 시작")
    recognizer.adjust_for_ambient_noise(source, duration=1)     # 주변 소음 조정
    audio = recognizer.listen(      # 마이크 입력 듣기
        source,                
        phrase_time_limit=5         # 녹음 최대 5초까지만
        )          

try:
    text2 = recognizer.recognize_google(audio, language='en')
    print(f"인식된 텍스트: {text2}")
except sr.UnknownValueError:
    print("음성을 이해하지 못했습니다. 다시 말해 주세요.")
except sr.RequestError as e:
    print(f"Google Speech Recognition API 요청 오류: {e}")

녹음 시작
음성을 이해하지 못했습니다. 다시 말해 주세요.


## 2. OpenAI API

- **gpt-4o-mini-transcribe** : OpenAI의 음성 인식 모델로, 오디오 파일을 텍스트로 변환하고 한국어 등 다양한 언어를 지원
- API 기반이라 로컬 모델 설치가 필요 없고, 비교적 간단한 코드로 전사를 수행할 수 있음

In [25]:
from dotenv import load_dotenv 

load_dotenv()

False

In [26]:
from openai import OpenAI

client = OpenAI()

In [ ]:
# https://developers.openai.com/api/docs/guides/audio

In [33]:
from pathlib import Path

audio_path = Path("./audio/my_voice.mp3")

with audio_path.open("rb") as f:
    result = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe",
        file=f,
        language="ko"
    )

print(result.text if hasattr(result, "text") else result)

안녕하세요. 오늘 비자 많이 옵니다. 날씨가 궂궂해요.


In [35]:
with sr.Microphone() as source:
    print("녹음 시작")
    recognizer.adjust_for_ambient_noise(source, duration=1)     # 주변 소음 조정
    # STEP1. 마이크 입력 받기
    audio = recognizer.listen(source)   
    # STEP2. 텍스트 변환
    print("인식 중입니다.....")
    text3 = recognizer.recognize_openai(audio)
    print(f"인식된 텍스트: {text3}")
    # STEP3. 오디오 저장
    audio_file = audio.get_wav_data()
    with open("./audio/real_voice.wav", "wb") as f:
        f.write(audio_file)
    print("목소리 저장완료")

녹음 시작
인식 중입니다.....
인식된 텍스트: チャンネル登録を是非お願いします。
목소리 저장완료


In [36]:
# 오디오 백그라운드 실행
from IPython.display import Audio, display
display(Audio("./audio/real_voice.wav", autoplay=True))

## 3. 로컬 Whisper 모델 

- **Whisper(base)** 는 OpenAI Whisper 모델을 로컬에서 실행해 음성을 텍스트로 변환하는 방식
- API 호출 없이 사용할 수 있어 실습과 오프라인 처리에 유리하지만, 모델 크기와 장비 성능에 따라 속도가 달라짐

In [34]:
%pip install openai-whisper

     ---------------------------------------- 0.0/803.2 kB ? eta -:--:--
     ---------------------------------------- 803.2/803.2 kB 6.2 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached numpy-2.4.6-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------------------------------------- -- 2.6/2.8 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 12.2 MB/s  0:00:00
   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
   ----- ---------------------------------- 5.5/41.9 MB 25.2 MB/s eta 0:00:02
   ---------- ----------------------------- 11.0/41.9 MB 26.4 MB/s eta 0:00:02


  You can safely remove it manually.
  You can safely remove it manually.


In [37]:
# whisper github 검색 
import whisper

# 모델 로드
model = whisper.load_model("base")

# 텍스트 변환
result = model.transcribe(
    "./audio/my_voice.mp3",
    language="ko",
    fp16=False)     # GPU 사용할거라면 True

print(result)
print(result["text"])

100%|███████████████████████████████████████| 139M/139M [00:08<00:00, 16.4MiB/s]


{'text': ' 안녕하세여 오늘 비가 많이 옵니다 날씨가 골고래요', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 4.0, 'text': ' 안녕하세여', 'tokens': [50364, 13810, 2240, 5762, 10558, 50564], 'temperature': 0.0, 'avg_logprob': -0.632027025575991, 'compression_ratio': 0.8625, 'no_speech_prob': 0.0882120206952095}, {'id': 1, 'seek': 0, 'start': 4.0, 'end': 10.0, 'text': ' 오늘 비가 많이 옵니다', 'tokens': [50564, 8880, 10079, 1453, 8358, 2355, 113, 1972, 50864], 'temperature': 0.0, 'avg_logprob': -0.632027025575991, 'compression_ratio': 0.8625, 'no_speech_prob': 0.0882120206952095}, {'id': 2, 'seek': 0, 'start': 10.0, 'end': 16.0, 'text': ' 날씨가 골고래요', 'tokens': [50864, 16316, 25416, 1453, 3352, 101, 1313, 167, 4241, 1495, 51164], 'temperature': 0.0, 'avg_logprob': -0.632027025575991, 'compression_ratio': 0.8625, 'no_speech_prob': 0.0882120206952095}], 'language': 'ko'}
 안녕하세여 오늘 비가 많이 옵니다 날씨가 골고래요


## 4. Faster-Whisper 모델

- **Faster-Whisper(base)** 는 Whisper를 CTranslate2 기반으로 최적화한 구현입니다.
- GPU와 `float16`을 활용하면 전사 속도와 메모리 효율이 좋아지고, 구간별 타임스탬프를 함께 다루기 쉬움

In [ ]:
# import os

# # Hugging Face Hub가 내 PC에 저장된 로그인 토큰을 자동으로 사용하지 말고, 공개 모델이면 그냥 익명으로 다운로드해라.
# os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# try:
#     import huggingface_hub.constants as hf_constants
#     hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
# except Exception:
#     pass

In [39]:
# faster-whisper github 검색
# uv add faster-whisper 
%pip install faster-whisper 
from faster_whisper import WhisperModel

model_size = "base"

# Run on GPU with FP16
model = WhisperModel(model_size, 
                     device="cpu", 
                     compute_type="int8")

# 텍스트 변환
segments, info = model.transcribe("./audio/my_voice.mp3",
                                  language='ko',
                                  beam_size=5)      # 높을 수록 정확하지만 느려진다.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.4.2 which is incompatible.


   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------ --------------------- 0.5/1.1 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.2 MB 2.7 MB/s eta 0:00:07
   ---- ----------------------------------- 2.1/19.2 MB 6.0 MB/s eta 0:00:03
   ------ --------------------------------- 2.9/19.2 MB 5.1 MB/s eta 0:00:04
   ------ --------------------------------- 2.9/19.2 MB 5.1 MB/s eta 0:00:04
   ------ --------------------------------- 3.1/19.2 MB 3.1 MB/s eta 0:00:06
   --------------- ------------------------ 7.6/19.2 MB 6.5 MB/s eta 0:00:02
   ------------------------ --------------- 11.5/19.2 MB 8.5 MB/s eta 0:00:01
   ---------------------------- ----------- 13.6/19.2 MB 8.7 MB/s eta 0:00:01
   ---------------------------- ----------- 13.9/19.2 MB 8.3 MB/s eta 0:00:01
   -------------

c:\Users\playdata2\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\playdata2\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\models--Systran--faster-whisper-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate develo

In [40]:
print(segments)
full_text = ""
for seg in segments:
    print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text}")
    full_text += seg.text
print(f"인식된 텍스트: {full_text}")
print(f"감지된 언어: {info.language} (확률: {info.language_probability:.0%})")

<generator object WhisperModel.generate_segments at 0x00000224F79511D0>
[0.00s -> 4.00s]  안녕하세여
[4.00s -> 10.00s]  오늘 피가 많이 옵니다
[10.00s -> 16.00s]  날씨가 골고해요
인식된 텍스트:  안녕하세여 오늘 피가 많이 옵니다 날씨가 골고해요
감지된 언어: ko (확률: 100%)


## [실습]
1. 짧은 음성 파일 STT 하기. 
   5초 이내의 짧은 음성 파일을 준비하고, `SpeechRecognition`으로 텍스트 변환 결과를 출력.
2. 한국어와 영어 인식 비교하기. 
   한국어 문장 1개와 영어 문장 1개를 각각 음성으로 준비하고, 언어 설정에 따라 인식 결과가 어떻게 달라지는지 비교.
3. Whisper와 OpenAI API 결과 비교하기. 같은 음성 파일을 로컬 Whisper와 OpenAI API로 각각 전사하고, 틀린 단어와 문장 부호 차이를 비교.
4. 회의록 자동 생성하기. 10초 이상 대화형 음성을 전사하고, 전사 결과를 요약해 `주제`, `핵심 내용`, `할 일` 형태로 정리.
5. STT 모델 성능 비교. `SpeechRecognition`, OpenAI STT, Whisper, Faster-Whisper 모델의 정확도, 속도 등을 비교.